In [31]:
import pandas as pd
import os
import numpy as np

In [8]:
def transform_dna_sequence(sequence):
    # Split sequence by spaces to get each triplet
    triplets = sequence.split()
    
    # Take the first character from the first triplet
    transformed_seq = triplets[0][0:3]
    
    # Concatenate with the last character of each subsequent triplet
    transformed_seq += ''.join([triplet[-1] for triplet in triplets[1:]])
    
    return transformed_seq


def transform_methyl_sequence(sequence):
    return "2"+sequence+"2" # 3mers were labeled for each mid-token, hence, padding orriginal sequence with 'unknown' metylation label for the first and last nucleotide

In [6]:
os.listdir("../Data/Raw/fig_2c")

['a0_b5', 'a1_b5', 'a2_b5', 'a3_b5']

In [ ]:
root = "../Data/Raw/fig_2c"
dest = root.replace("Raw", "Curated")
for dir in os.listdir("../Data/Raw/fig_2c"):
    for data_type in ["data", "test_data"]:

        data = pd.read_csv(os.path.join(root, dir, f"{data_type}.txt"), sep="\t")
        data['input_ids'] = data['dna_seq'].apply(transform_dna_sequence)
        data["methylation_ids"] = data["methyl_seq"].apply(transform_methyl_sequence)
        data['label'] = [int(x=="T") for x in data["ctype"]]
        data = data[["input_ids", "methylation_ids","label", "dmr_label"]]
        data.columns = ["input_ids", "methylation_ids","label", "dmr_id"]

        data["label"] = np.array(data["label"], dtype=np.int8)
        outdir = os.path.join(dest, dir)
        if not os.path.exists(outdir):
            os.mkdir(outdir)
        if(data_type == "data"):
            data.to_csv(os.path.join(outdir, "train.csv"),index=False)
        else:
            data.to_csv(os.path.join(outdir, "test.csv"),index=False)
            data.to_csv(os.path.join(outdir, "dev.csv"),index=False)

In [33]:
X = pd.read_csv("../Data/Curated/fig_2c/a3_b5/train.csv")

dtype('int64')